In [1]:
import polars as pl

import nwec.utility_reporting.arrearage_counts
import nwec.utility_reporting.arrearages
import nwec.utils.excel
from nwec.constants import RAW_UTILITY_DATA, Utility

YEAR = 2024
QUARTER = 4
NUM_MONTHS = 3
COLS_PER_MONTH = 5
SHEET_SEARCH_STRING = "Arrears"
ARREARAGE_SEARCH_STRING = "number of customers"
spreadsheet = RAW_UTILITY_DATA / str(YEAR) / f"{Utility.PAC.code}_{YEAR}_Q{QUARTER}.xlsx"
source_date_format = "%Y%m"

In [2]:
sheet_index = nwec.utils.excel.get_sheet_index_from_name(spreadsheet, SHEET_SEARCH_STRING)
df = pl.read_excel(spreadsheet, sheet_id=sheet_index, has_header=False)
arrearage_counts = nwec.utility_reporting.arrearages.get_arrearages_df(
    df, NUM_MONTHS, COLS_PER_MONTH, ARREARAGE_SEARCH_STRING
)

# Arrearage Counts


In [3]:
date_row = nwec.utility_reporting.arrearages.infer_date_row(arrearage_counts, source_date_format)
arrearage_counts = arrearage_counts.tail(-date_row)  # remove rows before the date row


In [4]:
months = arrearage_counts.slice(0, 1).to_dicts()[0]
months = [v for v in months.values() if v is not None]
if len(months) < NUM_MONTHS:
    months = nwec.utility_reporting.arrearages.extrapolate_missing_months(months, NUM_MONTHS, source_date_format)
assert len(months) == NUM_MONTHS, f"Expected {NUM_MONTHS} months, found {len(months)}."
vintages: pl.DataFrame = arrearage_counts.slice(1, 1)
for column in vintages:
    if str(column.first()).lower().strip() != "count":
        arrearage_counts = arrearage_counts.drop(column.name)
arrearage_counts = arrearage_counts.tail(-2)
month_row = pl.DataFrame(dict(zip(arrearage_counts.columns, months, strict=True)))
arrearage_counts = pl.concat([month_row, arrearage_counts])


In [ ]:
arrearage_counts = nwec.utility_reporting.arrearage_counts.format_arrearage_count_dates(
    arrearage_counts, source_date_format
)
arrearage_counts = nwec.utility_reporting.arrearages.add_zip_and_customer_class_cols(df, arrearage_counts)
arrearage_counts = nwec.utility_reporting.arrearage_counts.normalize_arrearage_count_cols(
    arrearage_counts, Utility.AVISTA
)


column_3,column_8,column_13
str,str,str
"""202410""","""202411""","""202412"""
"""114""","""155""","""87"""
"""188""","""190""","""198"""
"""90""","""151""","""116"""
null,null,null
…,…,…
"""3""","""9""","""5"""
"""34""","""67""","""74"""
"""96""","""81""","""67"""


2024 10,2024 11,2024 12
str,str,str
"""114""","""155""","""87"""
"""188""","""190""","""198"""
"""90""","""151""","""116"""
null,null,null
null,null,null
…,…,…
"""3""","""9""","""5"""
"""34""","""67""","""74"""
"""96""","""81""","""67"""


# Save Results


In [ ]:
nwec.utility_reporting.arrearage_counts.save_processed_arrearage_counts(arrearage_counts)